In [5]:
# Import necessary libraries
import pandas as pd
import numpy as np
import requests
import json
import time

In [6]:
# Load in site water level data dictionary
# Old dic that didnt register lat and long names
#with open('/capstone/aridgw/raw_data/site_coords.json', 'r') as f:
    #site_coords = {k: tuple(v) for k, v in json.load(f).items()}

#with open('/capstone/aridgw/raw_data/site_coords.json', 'r') as f:
    #site_coords = {k: (v["latitude"], v["longitude"]) for k, v in json.load(f).items()}

# Load in clean water level data csv that will be merged to
clean_site_data = "/capstone/aridgw/raw_data/clean_site_waterdata_05072026.csv"


In [7]:
df_sites = pd.read_csv(clean_site_data)
site_coords = (
    df_sites
    .drop_duplicates(subset="site_id")
    .set_index("site_id")[["latitude", "longitude"]]
    .apply(tuple, axis=1)
    .to_dict()
)
site_coords

{'KSGS.371852100505801': (37.31502, -100.8505),
 'KSGS.372043101363101': (37.34495, -101.6104),
 'KSGS.372539100142504': (37.42949, -100.2434),
 'KSGS.373331098033301': (37.56096, -98.058),
 'KSGS.373607100565301': (37.59874, -100.9497),
 'KSGS.374111099070401': (37.68448, -99.11925),
 'KSGS.374125100344101': (37.69024, -100.5795),
 'KSGS.374747100552101': (37.79492, -100.9225),
 'KSGS.375145100485701': (37.86209, -100.8179),
 'KSGS.375454101075401': (37.91572, -101.1306),
 'TWDB.0753701': (35.16, -102.4886),
 'TWDB.1005225': (34.97, -102.4381),
 'TWDB.1038101': (34.46, -102.3522),
 'TWDB.1039702': (34.38, -102.2336),
 'TWDB.1157901': (34.03, -101.9056),
 'TWDB.2409103': (33.8575, -102.9786),
 'TWDB.2409801': (33.76, -102.9564),
 'TWDB.2423701': (33.65222, -102.2278),
 'TWDB.2453902': (33.13833, -102.3808),
 'UNLCSD.423037099353801': (42.51028, -99.59429),
 'USGS.335258091152301': (33.88238, -91.25823),
 'USGS.340440091231201': (34.07647, -91.3878),
 'USGS.342649091251916': (34.46473, 

In [ ]:
# Split site_coords into two halves to avoid overloading the API
items = list(site_coords.items())
mid = len(items) // 2
first_half = dict(items[:mid])
second_half = dict(items[mid:])

# Use API to access OpenET Data

header = {"Authorization": "IeZA48KSBbWoU3a50ybYlaA77Qeal3H2UKLAA86KYJ9UKw94rXmcHKhxEOTg"}

# Select boundary box side length
size_km = 10

# Extract data sections less than 10 years as required by the API
date_ranges = [
    ("2000-01-01", "2009-12-31", 2.0), 
    ("2010-01-01", "2019-12-31", 2.0),
    ("2020-01-01", "2020-12-31", 2.1), # Use version 2.1 for newer versions which may not have accurate data for older versions
]

dfs = [] # Create empty dataframe

# Create function to generate boundary box given coordinates and boundary box size
def make_square_geometry(lat, lon, size_km):
    half = size_km / 2
    lat_offset = half / 111.0
    lon_offset = half / (111.0 * np.cos(np.radians(lat)))
    return [
        lon - lon_offset, lat + lat_offset,
        lon - lon_offset, lat - lat_offset,
        lon + lon_offset, lat - lat_offset,
        lon + lon_offset, lat + lat_offset,
        lon - lon_offset, lat + lat_offset,
    ]

# Retry failed requests to handle intermittent server errors
def query_with_retry(header, polygon_args, retries=5, wait=5):
    for attempt in range(retries):
        resp = requests.post(
            headers=header,
            json=polygon_args,
            url="https://openet-api.org/raster/timeseries/polygon"
        )
        if resp.status_code == 200:
            return resp
        print(f"    Attempt {attempt + 1} failed — retrying in {wait}s...")
        time.sleep(wait)
    return resp

# Use API to extract ET data, wrapped in a function to allow querying in two batches
def query_sites(sites, dfs):
    for name, (lat, lon) in sites.items():
        print(f"Querying {name}...")
        
        for start_date, end_date, version in date_ranges:

            polygon_args = { # Define query parameters
                "date_range": [start_date, end_date],
                "interval": "monthly",
                "geometry": make_square_geometry(lat, lon, size_km),
                "model": "Ensemble",
                "reducer": "mean",
                "variable": "ET",
                "reference_et": "gridMET",
                "units": "mm",
                "file_format": "JSON",
                "version": version
            }

            resp = query_with_retry(header, polygon_args) # Request polygon API with retry logic

            print(f"  {name} {start_date}–{end_date} — Status: {resp.status_code}") # Report status of data to console

            if resp.status_code != 200: # Return error if unable to extract data
                print(f"  ERROR: {resp.json()}")
                continue

            data = resp.json()
            df = pd.DataFrame(data)

            # Add columns to dataframe to specify site specific information
            df["site"] = name
            df["latitude"] = lat
            df["longitude"] = lon
            df["bbox_side"] = size_km
            df["open_et_version"] = version
            dfs.append(df)
    return dfs

# Query first half of sites, pause, then query second half
dfs = query_sites(second_half, dfs)
print("First half done — pausing before second half...")
time.sleep(10)
dfs = query_sites(second_half, dfs)

df_monthly = pd.concat(dfs, ignore_index=True) # Add new data as new observations in dataframe
print(df_monthly.head())

Querying USGS.373644113411501...
  USGS.373644113411501 2000-01-01–2009-12-31 — Status: 200
    Attempt 1 failed — retrying in 5s...
  USGS.373644113411501 2010-01-01–2019-12-31 — Status: 200
    Attempt 1 failed — retrying in 5s...
  USGS.373644113411501 2020-01-01–2020-12-31 — Status: 200
Querying USGS.381901113014102...
    Attempt 1 failed — retrying in 5s...
    Attempt 2 failed — retrying in 5s...
  USGS.381901113014102 2000-01-01–2009-12-31 — Status: 200
  USGS.381901113014102 2010-01-01–2019-12-31 — Status: 200
  USGS.381901113014102 2020-01-01–2020-12-31 — Status: 200
Querying USGS.383921112175901...
  USGS.383921112175901 2000-01-01–2009-12-31 — Status: 200
  USGS.383921112175901 2010-01-01–2019-12-31 — Status: 200
  USGS.383921112175901 2020-01-01–2020-12-31 — Status: 200
Querying USGS.390819111530701...
  USGS.390819111530701 2000-01-01–2009-12-31 — Status: 200
  USGS.390819111530701 2010-01-01–2019-12-31 — Status: 200
    Attempt 1 failed — retrying in 5s...
    Attempt 2 

In [ ]:
#final_df = pd.concat([df_monthly_concat, df_monthly])

In [ ]:

#final_df.to_csv("monthly_et_10km.csv", index=False)

In [ ]:

# df_monthly_first_half = df_monthly
# df_monthly_first_half.to_csv("df_monthly_first_half.csv", index=False)

In [ ]:
# Find sites with incomplete data 
#final_df.groupby('site').size().reset_index(name='count').query('count < 312')#.ungroup()

,site,count
0,KSGS.371852100505801,252
1,KSGS.372043101363101,252
2,KSGS.372539100142504,252
3,KSGS.373331098033301,252
4,KSGS.373607100565301,252
5,KSGS.374111099070401,252
6,KSGS.374125100344101,252
7,KSGS.374747100552101,252
8,KSGS.375145100485701,252
9,KSGS.375454101075401,252


In [ ]:
# Convert time column to datetime, extract year, rename et column
df_monthly["time"] = pd.to_datetime(df_monthly["time"], format='%Y-%m-%d')
df_monthly["year"] = df_monthly["time"].dt.year
df_monthly = df_monthly.rename(columns={"et": "monthly_et_avg"})

In [87]:
# Check number of monthly et observations per site
# Ensure value consistency
df_monthly.groupby(["site"]).size().reset_index(name="n")

,site,n
0,KSGS.371852100505801,252
1,KSGS.372043101363101,252
2,KSGS.372539100142504,252
3,KSGS.373331098033301,252
4,KSGS.373607100565301,252
5,KSGS.374111099070401,252
6,KSGS.374125100344101,252
7,KSGS.374747100552101,252
8,KSGS.375145100485701,252
9,KSGS.375454101075401,252


In [90]:
# Convert monthly ET data into csv
df_monthly.to_csv(f"/capstone/aridgw/raw_data/open_et/{size_km}km/openet_data_monthly_{size_km}km.csv", index=False)

In [91]:
# Calculate a single monthly average per year 
df_annual = (
    df_monthly
    .groupby(["site", "year", "latitude", "longitude", "bbox_side", "open_et_version"])['monthly_et_avg']
    .mean()
    .reset_index()
    .rename(columns={"monthly_et_avg": "annual_et_avg"})
)

In [92]:
# Rescale annual et 
df_annual["scaled_annual_et_avg"] = (df_annual["annual_et_avg"] * 12).round(3)
df_annual = df_annual.drop(columns=["annual_et_avg"])


In [93]:
# Check number of scaled_annual_et_avg values per year
# Ensure value consistency
df_annual.groupby(["year"]).size().reset_index(name="n") 

,year,n
0,2000,50
1,2001,50
2,2002,50
3,2003,50
4,2004,50
5,2005,50
6,2006,50
7,2007,50
8,2008,50
9,2009,50


In [94]:
# Convert annual ET data into csv
df_annual.to_csv(f"/capstone/aridgw/raw_data/open_et/{size_km}km/openet_data_annual_{size_km}km.csv", index=False)

In [95]:
# Remove lat and long to above columns duplicates to prepare for merge 
df_annual = df_annual.drop(columns=["latitude", "longitude"])

In [96]:
df_annual

,site,year,bbox_side,open_et_version,scaled_annual_et_avg
0,KSGS.371852100505801,2000,10,2.0,682.922
1,KSGS.371852100505801,2001,10,2.0,666.249
2,KSGS.371852100505801,2002,10,2.0,631.744
3,KSGS.371852100505801,2003,10,2.0,722.681
4,KSGS.371852100505801,2004,10,2.0,686.214
...,...,...,...,...,...
1045,willcox,2016,10,2.0,548.820
1046,willcox,2017,10,2.0,493.566
1047,willcox,2018,10,2.0,528.623
1048,willcox,2019,10,2.0,550.907


In [97]:
df_sites.groupby(["site_id"]).size().reset_index(name="n") 

,site_id,n
0,KSGS.371852100505801,21
1,KSGS.372043101363101,21
2,KSGS.372539100142504,21
3,KSGS.373331098033301,21
4,KSGS.373607100565301,21
5,KSGS.374111099070401,21
6,KSGS.374125100344101,21
7,KSGS.374747100552101,21
8,KSGS.375145100485701,21
9,KSGS.375454101075401,21


In [98]:
# Merge ET data to water level data 
df_merged_et = pd.merge(
    df_sites,
    df_annual,
    left_on=['site_id', 'year_value'],
    right_on=['site', 'year'],
    how='outer'
)

In [99]:
# Remove rows containing NA in columns important for analysis
df_merged_et = df_merged_et.dropna(subset=["depth_to_gw_m", "scaled_annual_et_avg"])

In [100]:
# Change year_value to int
df_merged_et["year_value"] = df_merged_et["year_value"].astype(int)

In [101]:
# Check number of observations per site in merged df
df_merged_et.groupby(["site_id"]).size().reset_index(name="n")

,site_id,n
0,KSGS.371852100505801,21
1,KSGS.372043101363101,21
2,KSGS.372539100142504,21
3,KSGS.373331098033301,21
4,KSGS.373607100565301,21
5,KSGS.374111099070401,21
6,KSGS.374125100344101,21
7,KSGS.374747100552101,21
8,KSGS.375145100485701,21
9,KSGS.375454101075401,21


In [102]:
# Remove duplicate year column
df_merged_et = df_merged_et.drop(columns='year')
df_merged_et = df_merged_et.drop(columns='site')

In [103]:
df_merged_et

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,bbox_side,open_et_version,scaled_annual_et_avg
0,2000,KSGS.371852100505801,239.390000,72.966072,37.31502,-100.8505,USGS,Southern Kansas,10,2.0,682.922
1,2001,KSGS.371852100505801,241.960000,73.749408,37.31502,-100.8505,USGS,Southern Kansas,10,2.0,666.249
2,2002,KSGS.371852100505801,242.780000,73.999344,37.31502,-100.8505,USGS,Southern Kansas,10,2.0,631.744
3,2003,KSGS.371852100505801,246.710000,75.197208,37.31502,-100.8505,USGS,Southern Kansas,10,2.0,722.681
4,2004,KSGS.371852100505801,247.710000,75.502008,37.31502,-100.8505,USGS,Southern Kansas,10,2.0,686.214
...,...,...,...,...,...,...,...,...,...,...,...
1043,2014,willcox,346.899946,105.735100,32.03619,-109.7540,Jasechko,Southwest US,10,2.0,509.436
1044,2015,willcox,352.000011,107.289600,32.03619,-109.7540,Jasechko,Southwest US,10,2.0,568.491
1046,2017,willcox,372.750012,113.614200,32.03619,-109.7540,Jasechko,Southwest US,10,2.0,493.566
1048,2019,willcox,392.100078,119.512100,32.03619,-109.7540,Jasechko,Southwest US,10,2.0,550.907


In [104]:
# Turn merged df into csv
# Convert monthly ET data into csv
df_merged_et.to_csv(f"/capstone/aridgw/outputs/{size_km}km/gwl_cult_et_{size_km}km.csv", index=False)